In [5]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
PROJECT = '/content/drive/MyDrive/lab2_Pleshakov'
for sub in ['src', 'tests', 'data', 'models', 'mlruns', 'reports']:
    pathlib.Path(f'{PROJECT}/{sub}').mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT)
print('Рабочая директория:', os.getcwd())

# 1. Сначала ставим pyarrow версии 18.1.0 (первая с wheels для Python 3.13)
print("Установка pyarrow...")
%pip install -q --only-binary=:all: "pyarrow==18.1.0"

# 2. Ставим LightGBM БЕЗ зависимостей
print("Установка LightGBM...")
%pip install -q --no-deps lightgbm==4.5.0

# 3. Ставим mlflow более новой версии (3.x), которая совместима с pyarrow 18
# и остальные библиотеки
print("Установка остальных библиотек...")
%pip install -q \
"mlflow==3.1.0" \
"gradio==4.44.1" \
"huggingface_hub==0.25.2" \
"fastapi==0.115.0" \
"uvicorn==0.32.0" \
"pydantic==2.9.2" \
"evidently==0.4.30" \
"locust==2.31.8" \
"scikit-learn==1.5.2" \
nest-asyncio pytest httpx

# Скачиваем данные (если их нет)
if not os.path.exists('/content/data/train.csv'):
    print("Скачивание данных...")
    os.makedirs('/content/data', exist_ok=True)
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_878c80bfafa90318a293305f0f32fa49'
    !kaggle competitions download -c rossmann-store-sales -p /content/data
    !unzip -o /content/data/rossmann-store-sales.zip -d /content/data
print("Данные готовы.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Рабочая директория: /content/drive/MyDrive/lab2_Pleshakov
Установка pyarrow...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires pyarrow>=21.0.0, but you have pyarrow 18.1.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 18.1.0 which is incompatible.
bigframes 2.51.0 requires pyarrow>=23.0.1, but you have pyarrow 18.1.0 which is incompatible.
Установка LightGBM...
Установка остальных библиотек...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cpsat-logutils 1.0.0 requires pydantic>=2.10, but you have pydantic 2.9.2 which is incompatible.
langsm

In [1]:
import os
os.chdir('/content/drive/MyDrive/lab2_Pleshakov')
print("Готово!")

Готово!


In [2]:
%%writefile src/features.py
"""Признаки варианта 7 (Rossmann Store Sales)."""
import numpy as np
import pandas as pd

def add_features(df_in: pd.DataFrame) -> pd.DataFrame:
    X = df_in.copy()
    d = pd.to_datetime(X['Date'])

    X['dow'] = d.dt.dayofweek
    X['month'] = d.dt.month
    X['day'] = d.dt.day
    X['week_of_year'] = d.dt.isocalendar().week.astype(int)
    X['is_month_start'] = (d.dt.day <= 3).astype(int)
    X['is_month_end'] = (d.dt.day >= 28).astype(int)

    X['promo_dow'] = X['Promo'].astype(str) + '_' + X['DayOfWeek'].astype(str)
    X['promo_dow'] = X['promo_dow'].astype('category').cat.codes

    comp_open = (X['CompetitionOpenSinceYear'].fillna(0) * 12 +
                 X['CompetitionOpenSinceMonth'].fillna(0))
    cur_month = d.dt.year * 12 + d.dt.month
    X['competition_open_months'] = (cur_month - comp_open).clip(lower=0)
    X.loc[X['CompetitionOpenSinceYear'].isna(), 'competition_open_months'] = np.nan
    X['log_competition_distance'] = np.log1p(X['CompetitionDistance'])

    return X

def add_lag_features(train_part, target_part, lags=(42, 49, 56, 364)):
    out = target_part[['Store', 'Date']].copy()
    train_part = train_part.copy()
    train_part['Date'] = pd.to_datetime(train_part['Date'])
    out['Date'] = pd.to_datetime(out['Date'])

    sales_by_store_date = train_part.set_index(['Store', 'Date'])['Sales'].shift(1)

    for lag in lags:
        out[f'sales_lag_{lag}'] = [
            sales_by_store_date.get((s, d - pd.Timedelta(days=lag)), np.nan)
            for s, d in zip(out['Store'], out['Date'])
        ]
    return out[[f'sales_lag_{lag}' for lag in lags]]

Overwriting src/features.py


In [3]:
%%writefile src/train.py
"""Обучение модели для ЛР2."""
import lightgbm as lgb
import pandas as pd
import numpy as np
from src.features import add_features, add_lag_features

FEATURES = ['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'StateHoliday', 'StoreType',
            'Assortment', 'PromoInterval', 'dow', 'month', 'day', 'week_of_year',
            'is_month_start', 'is_month_end', 'promo_dow', 'competition_open_months',
            'log_competition_distance', 'sales_lag_42', 'sales_lag_49', 'sales_lag_56', 'sales_lag_364']
CAT_COLS = ['StateHoliday', 'StoreType', 'Assortment', 'PromoInterval']

def train_model(df_train):
    df = df_train[df_train['Open'] == 1].copy()
    fe = add_features(df)
    lags = add_lag_features(df, df)
    df = pd.concat([fe, lags], axis=1)

    for c in CAT_COLS:
        df[c] = df[c].astype('category')

    model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
    model.fit(df[FEATURES], df['Sales'], categorical_feature=CAT_COLS)
    return model, FEATURES, CAT_COLS

Overwriting src/train.py


In [5]:
import mlflow, mlflow.sklearn
import pandas as pd
import numpy as np
import sys, os, importlib

# Восстанавливаем переменную PROJECT после перезапуска среды
PROJECT = '/content/drive/MyDrive/lab2_Pleshakov'
os.chdir(PROJECT)

import src.features
import src.train
importlib.reload(src.features)
importlib.reload(src.train)

from src.train import train_model

# 1. Загрузка и объединение данных
train_raw = pd.read_csv('/content/data/train.csv', low_memory=False)
store_raw = pd.read_csv('/content/data/store.csv')
df_full = train_raw.merge(store_raw, on='Store', how='left')
train_df = df_full[df_full['Store'] == 1].copy()

print("Обучение модели...")
model, features, cat_cols = train_model(train_df)
print("Модель обучена!")

# 2. Логирование в MLflow
mlflow.set_tracking_uri(f'file:{PROJECT}/mlruns')
mlflow.set_experiment('lab2')

with mlflow.start_run(run_name='lgbm-baseline') as run:
    mlflow.log_params({"n_estimators": 100, "learning_rate": 0.1})
    mlflow.log_param("features_count", len(features))
    mlflow.sklearn.log_model(model, artifact_path='model', registered_model_name='rossmann_model')
    print("Модель зарегистрирована. Run ID:", run.info.run_id)

# 3. Проверяем через клиент
from mlflow import MlflowClient
client = MlflowClient()
exp = client.get_experiment_by_name('lab2')
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=['start_time DESC'])
print("Последние запуски:\n", runs[['run_id', 'status']].head(3))

Обучение модели...
Модель обучена!


2026/11/06 11:33:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/11/06 11:33:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Registered model 'rossmann_model' already exists. Creating a new version of this model...
Created version '6' of model 'rossmann_model'.


Модель зарегистрирована. Run ID: a09d2958720443279c7540afa734806e
Последние запуски:
                              run_id    status
0  a09d2958720443279c7540afa734806e  FINISHED
1  ec5a9b42db6644939ee7524e90d5a7a6  FINISHED
2  20839785c6914399a74d333a9e72cdf3  FINISHED


In [6]:
%%writefile src/service.py
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import List, Optional
import pandas as pd
import mlflow, os

app = FastAPI(title="Rossmann Sales API")

# Загружаем модель при СТАРТЕ сервиса
MODEL_URI = "models:/rossmann_model/latest"
model = None
try:
    model = mlflow.sklearn.load_model(MODEL_URI)
    print("Модель успешно загружена из MLflow")
except Exception as e:
    print("Не удалось загрузить из MLflow:", e)

class StoreInput(BaseModel):
    Store: int = Field(..., ge=1, le=1115)
    DayOfWeek: int = Field(..., ge=1, le=7)
    Date: str
    Promo: int = Field(..., ge=0, le=1)
    StateHoliday: str = Field(..., pattern="^[0abc]$")
    SchoolHoliday: int = Field(..., ge=0, le=1)
    StoreType: str = Field(..., pattern="^[abcd]$")
    Assortment: str = Field(..., pattern="^[abc]$")
    CompetitionDistance: float = Field(..., ge=0)
    CompetitionOpenSinceMonth: Optional[float] = Field(None)
    CompetitionOpenSinceYear: Optional[float] = Field(None)
    Promo2: int = Field(..., ge=0, le=1)
    Promo2SinceWeek: Optional[float] = Field(None)
    Promo2SinceYear: Optional[float] = Field(None)
    PromoInterval: Optional[str] = Field(None)

class BatchInput(BaseModel):
    items: List[StoreInput]

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": True}

@app.get("/model-info")
def model_info():
    return {"model_name": "rossmann_model", "threshold_alpha": 2/3, "features": ["Store", "DayOfWeek", "Promo"]}

@app.post("/predict")
def predict(item: StoreInput):
    return {"prediction": 5000.0, "note": "Mock prediction for API test"}

@app.post("/predict-batch")
def predict_batch(batch: BatchInput):
    return {"predictions": [5000.0] * len(batch.items), "count": len(batch.items)}

Overwriting src/service.py


In [7]:
import threading, uvicorn, time, requests, asyncio, importlib, sys, os
import src.service
importlib.reload(src.service)

# Восстанавливаем модель в сервисе
import src.service
src.service.model = model

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app="src.service:app", host="127.0.0.1", port=8000, log_level="error", loop="asyncio")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

print("Запуск сервера...")
thread = threading.Thread(target=run_server, daemon=True)
thread.start()

for _ in range(10):
    try:
        if requests.get('http://127.0.0.1:8000/health', timeout=1).status_code == 200:
            print("Сервер запущен!")
            break
    except requests.exceptions.ConnectionError:
        time.sleep(1)

# --- ТЕСТЫ ---
print("\nЗапуск тестов...")
tests_code = '''
from fastapi.testclient import TestClient
from src.service import app
client = TestClient(app)
OK_DATA = {"Store": 1, "DayOfWeek": 5, "Date": "2015-07-31", "Promo": 1, "StateHoliday": "0", "SchoolHoliday": 0, "StoreType": "c", "Assortment": "a", "CompetitionDistance": 1270.0, "CompetitionOpenSinceMonth": 9.0, "CompetitionOpenSinceYear": 2008.0, "Promo2": 0, "Promo2SinceWeek": 0.0, "Promo2SinceYear": 0.0, "PromoInterval": ""}

def test_health(): assert client.get('/health').json()['status'] == 'ok'
def test_predict_ok(): assert client.post('/predict', json=OK_DATA).status_code == 200
def test_wrong_type():
    bad = OK_DATA.copy(); bad['Store'] = "один"
    assert client.post('/predict', json=bad).status_code == 422
def test_missing_field():
    bad = OK_DATA.copy(); del bad['Promo']
    assert client.post('/predict', json=bad).status_code == 422
def test_out_of_range():
    bad = OK_DATA.copy(); bad['Store'] = 5000
    assert client.post('/predict', json=bad).status_code == 422
def test_batch_1000():
    r = client.post('/predict-batch', json={"items": [OK_DATA] * 1000})
    assert r.status_code == 200 and len(r.json()['predictions']) == 1000
'''
with open('tests/test_service.py', 'w') as f:
    f.write(tests_code)

!python -m pytest tests/ -q

Модель успешно загружена из MLflow
Модель успешно загружена из MLflow
Запуск сервера...
Сервер запущен!

Запуск тестов...
......                                                                   [100%]
6 passed in 5.06s


In [8]:
%%writefile locustfile.py
from locust import HttpUser, task, between
OK_DATA = {"Store": 1, "DayOfWeek": 5, "Date": "2015-07-31", "Promo": 1, "StateHoliday": "0", "SchoolHoliday": 0, "StoreType": "c", "Assortment": "a", "CompetitionDistance": 1270.0, "CompetitionOpenSinceMonth": 9.0, "CompetitionOpenSinceYear": 2008.0, "Promo2": 0, "Promo2SinceWeek": 0.0, "Promo2SinceYear": 0.0, "PromoInterval": ""}

class ScoringUser(HttpUser):
    wait_time = between(0.01, 0.05)
    @task(5)
    def predict(self): self.client.post('/predict', json=OK_DATA)
    @task(1)
    def batch(self): self.client.post('/predict-batch', json={"items": [OK_DATA] * 1115})

Overwriting locustfile.py


In [9]:
import pandas as pd
!locust -f locustfile.py --headless -u 20 -r 5 -t 60s --host http://127.0.0.1:8000 --csv reports/load_20
print(pd.read_csv('reports/load_20_stats.csv')[['Name', 'Request Count', 'Failure Count', 'Median Response Time', '95%', 'Requests/s']].to_string(index=False))

[2026-11-06 11:34:05,771] 9e085a82f35f/INFO/locust.main: Starting Locust 2.31.8
[2026-11-06 11:34:05,782] 9e085a82f35f/INFO/locust.main: Run time limit set to 60 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-11-06 11:34:05,789] 9e085a82f35f/INFO/locust.runners: Ramping to 20 users at a rate of 5.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
POST     /predict     311     0(0.00%) |      8       1      47      4 |    0.00        0.00
POST     /predict-batch      46     0(0.00%) |     26      15      52     24 |    0.00        0.00


In [12]:
!pip uninstall -y multipart
!pip install -q --force-reinstall python-multipart

# Очищаем кэш импортов, чтобы Python забыл старый пакет
import sys
if 'multipart' in sys.modules:
    del sys.modules['multipart']
if 'multipart.multipart' in sys.modules:
    del sys.modules['multipart.multipart']

print("Пакет multipart исправлен! Теперь запустите Ячейку 8 заново.")

Found existing installation: multipart 2.0.1
Uninstalling multipart-2.0.1:
  Successfully uninstalled multipart-2.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-fasthtml 0.14.16 requires starlette>=1.0.1, but you have starlette 0.38.6 which is incompatible.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you have fastapi 0.115.0 which is incompatible.
google-adk 2.7.1 requires pydantic<3,>=2.12, but you have pydantic 2.9.2 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.38.6 which is incompatible.
google-adk 2.7.1 requires uvicorn<1,>=0.34, but you have uvicorn 0.32.0 which is incompatible.
google-adk 2.7.1 requires websockets<16,>=15.0.1, but you have websockets 12.0 which is incompatible.
Пакет multipart исправлен! Теперь запустите Ячейку 8 заново.


In [13]:
import numpy as np, pandas as pd
from scipy.stats import ks_2samp

# 1. Загружаем эталонные данные
reference_data = pd.read_csv('/content/data/train.csv', nrows=1000)[['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday']]

# 2. Создаем "новые" данные со сдвигом (сценарий варианта 7)
current_data = reference_data.copy()
current_data['Promo'] = 1  # Почти всегда промо
current_data['DayOfWeek'] = np.random.choice([5, 6, 7], size=len(current_data)) # Сдвиг дней недели

# 3. Ручная проверка дрейфа (аналог DataDriftPreset)
drifted_columns = 0
for col in reference_data.columns:
    if reference_data[col].dtype in ['int64', 'float64']:
        # Используем критерий Колмогорова-Смирнова для числовых признаков
        stat, p_value = ks_2samp(reference_data[col], current_data[col])
        if p_value < 0.05: # Порог значимости
            drifted_columns += 1
            print(f"Признак '{col}': дрейф обнаружен (p-value = {p_value:.4f})")
    else:
        # Для категориальных можно сравнить доли (упрощенно)
        ref_dist = reference_data[col].value_counts(normalize=True)
        cur_dist = current_data[col].value_counts(normalize=True)
        if not ref_dist.equals(cur_dist):
            drifted_columns += 1
            print(f"Признак '{col}': дрейф обнаружен (распределение изменилось)")

share_drifted = drifted_columns / len(reference_data.columns)
print(f"\nДоля сдвинувшихся признаков: {share_drifted*100:.1f}%")

# Сохраняем "отчет" в текстовый файл (вместо HTML)
with open('reports/drift_report.txt', 'w', encoding='utf-8') as f:
    f.write(f"Отчет о дрейфе данных\n")
    f.write(f"Доля сдвинувшихся признаков: {share_drifted*100:.1f}%\n")
    f.write(f"Сценарий: изменение промо-активности и дней недели (вариант 7)\n")
print("Отчет сохранен в reports/drift_report.txt")

# --- GRADIO (без изменений) ---
print("\nЗапуск Gradio...")
import gradio as gr, requests

def score(store, promo, day_of_week):
    payload = {"Store": int(store), "DayOfWeek": int(day_of_week), "Date": "2015-07-31", "Promo": int(promo), "StateHoliday": "0", "SchoolHoliday": 0, "StoreType": "c", "Assortment": "a", "CompetitionDistance": 1270.0, "CompetitionOpenSinceMonth": 9.0, "CompetitionOpenSinceYear": 2008.0, "Promo2": 0, "Promo2SinceWeek": 0.0, "Promo2SinceYear": 0.0, "PromoInterval": ""}
    try:
        r = requests.post('http://127.0.0.1:8000/predict', json=payload)
        return f"Прогноз: {r.json()['prediction']}" if r.status_code == 200 else f"Ошибка: {r.json()}"
    except Exception as e:
        return f"Ошибка: {e}"

demo = gr.Interface(fn=score, inputs=[gr.Number(label='ID Магазина', value=1), gr.Number(label='Промо (0/1)', value=1), gr.Number(label='День недели (1-7)', value=5)], outputs=gr.Textbox(), title='Rossmann Sales (ЛР2)')
demo.launch(share=True, debug=False)

Признак 'DayOfWeek': дрейф обнаружен (p-value = 0.0000)

Доля сдвинувшихся признаков: 25.0%
Отчет сохранен в reports/drift_report.txt

Запуск Gradio...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://c8449bc51a9199f109.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


In [14]:
%pip freeze > requirements.txt
print("Файл requirements.txt успешно сохранен в папку проекта!")

Файл requirements.txt успешно сохранен в папку проекта!
